### What is Agentic AI?
Agentic AI refers to AI systems designed with autonomous decision-making capabilities, enabling them to perceive, reason, and take actions dynamically. These systems leverage multi-agent architectures,and LLMs to perform complex tasks with minimal human intervention. They are widely used in automation, cybersecurity, finance, and enterprise AI for intelligent workflows. Tools like LangChain, CrewAI, AutoGen, and LlamaIndex facilitate building and orchestrating these AI agents.

<center><img src="https://cdn.prod.website-files.com/66c435ec15ed715aec9ee3fe/66fd1383a0e470c48179a973_66fd136aff11aa6797621e82_What%2520is%2520agentic%2520AI.jpeg" width=600/></center>

### How is it different from Generative AI?

<center><img src="https://cdn.prod.website-files.com/66c435ec15ed715aec9ee3fe/66e78a95f15da5104e75ac08_66e78a880dcb8cb0f1229412_Agentic%2520AI%2520vs%2520Generative%2520AI.png" width =600/></center>

# Generative AI basics:
* ## Transformers
* ## LLMs


In [ ]:
# Installing the LangChain Hub package to access and manage pre-built AI chains, prompts, and agents.
!pip install langchainhub

# Installing the LangChain OpenAI integration to use OpenAI models within LangChain workflows.
!pip install langchain-openai

# Installing the core LangChain library for building LLM-based applications, including chaining, memory, and retrieval capabilities.
!pip install langchain

# Installing the community version of LangChain, which includes integrations and tools contributed by the community.
!pip install langchain-community

# Installing FAISS (Facebook AI Similarity Search) for efficient similarity-based search on text embeddings.
!pip install faiss-cpu

# Installing Gradio, a framework to create web-based UIs for AI models and applications easily.
!pip install gradio

## Basics of RAG

Before we start coding, lets go over a few questions and get it clarified.

### 1. What is RAG?
Imagine you're writing an article about climate change, but instead of relying only on what you remember, you search online for recent studies and data to support your writing. RAG works the same way—it combines a powerful AI language model with a search system that retrieves relevant information from an external knowledge source, like a database or documents. This helps generate more accurate and informative responses.

### 2. Why is RAG important?
RAG is crucial because AI models, like ChatGPT, can sometimes "hallucinate" or provide outdated or incorrect information. By retrieving facts from trusted sources before generating responses, RAG ensures the answers are more reliable, up-to-date, and contextually relevant.

### 3. What’s the difference between RAG and a standard AI chatbot?
A standard chatbot relies only on pre-trained knowledge, which may be limited or outdated. RAG-enhanced chatbots, however, actively retrieve fresh, relevant information from external sources, ensuring better accuracy and up-to-date insights.

### 4. What are the key components of RAG?
RAG consists of two main parts:

- Retriever: Finds the most relevant documents or data from a knowledge base (e.g., a search engine or database).
- Generator: Uses the retrieved information to produce a coherent and accurate response.

### 5. Usecases in real-life
- Customer Support: AI chatbots retrieve knowledge base articles to provide better responses to customer queries.
- Healthcare: Doctors can get AI-assisted summaries of patient records and the latest medical research.
- Legal Services: Lawyers can search through legal documents and case studies to build stronger cases.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Understanding the Limitation of the LLM

In [ ]:
# Importing the OpenAI library to interact with OpenAI's API services.
from openai import OpenAI

In [ ]:
import os  # Importing the os module to interact with environment variables
import getpass  # Importing getpass to securely input sensitive information

# Prompting the user to securely enter their OpenAI API key without displaying it on the screen
OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")

In [ ]:
# Creating an instance of the OpenAI client using the provided API key.
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# Defining the prompt to query the LLM
prompt = ''' What was uber's revenue in 2022? '''

In [ ]:
# Sending a request to the OpenAI API to generate a chat response
openai_response = client.chat.completions.create(
    model='gpt-3.5-turbo',  # Specifying the model to use;
    # Note: An older model chosen for testing purposes because the cutoff is 2021 whereas prompt is querying details about 2022
    messages=[{'role': 'user', 'content': prompt}]  # Creating a structured message for the AI model
)


In the above code, while creating a structured message for the AI model,  `role `defines the speaker (user input) and `content` contains the actual query stored in the `prompt` variable.

We are structuring the input this way because OpenAI's chat models require a specific format to understand and process conversations effectively. Assigning roles like 'user' helps the AI distinguish between different participants in the conversation, ensuring it provides relevant and context-aware responses.

In [ ]:
# Accessing the generated response from the AI model.
openai_response.choices[0].message.content
# Note:'choices' contains multiple response options, we take the first one ([0]),
# 'message' holds the response details, and 'content' extracts the actual text generated by the AI.

# We should expect that it will refuse to answer this -- due to knowledge cutoff.
# We're intentionally using an old model to demonstrate this.

### Interpretation:
Why is the LLM not able to answer the query?

`gpt-3.5-turbo` does not have access to data after 2021 because of its cutoff.

#### Do it yourself:
Try changing the model to `gpt-4o-mini` and observe how the output changes!

In [ ]:
# Let's try!

# Let's see if the newer 4o-mini will answer.
openai_response = client.chat.completions.create(
    model='gpt-4o-mini',  # <-- Here.
    messages=[{'role': 'user', 'content': prompt}]
)

openai_response.choices[0].message.content

# NOTE: We should expect that it does indeed answer.
# Leter model -- later knowledge cutoff.

As you can see above, the LLM we used(gpt 3.5) doesn't have access to the latest data. Now as LLMs get updated, the training cut-off date may or may not have access to more information. However, it's always a good idea to understand how to improve the context of our prompt

### Making the LLM context-aware

Next Step: Let's check Uber's [financial report ](https://s23.q4cdn.com/407969754/files/doc_events/2024/May/06/2023-annual-report.pdf)

On Page 54, of the above document it states:

"Revenue was 37.3 billion, up 17% year-over-year. Mobility revenue increased 5.8 billion primarily attributable to an increase in
Mobility Gross Bookings of......"

In [ ]:
## Let's create the above context for the prompt (manually, for now)
# Defining a context string with revenue details retrieved from an external source.
retrieved_context = '''Revenue was $37.3 billion, up 17% year-over-year. Mobility revenue increased $5.8 billion primarily attributable to an increase in
               Mobility Gross Bookings of 31% year-over-year.'''

In [ ]:
## Let's modify our prompt now
# Creating a prompt by embedding the retrieved context into a question for the AI model.

prompt = f"What was Uber's revenue in 2022? Check in {retrieved_context}"

# Note: The AI is being asked to analyze the given context and provide Uber's revenue for 2022

In [ ]:
## Let's ask the LLM again
openai_response = client.chat.completions.create(
    model = 'gpt-3.5-turbo',
    messages = [{'role': 'user', 'content': prompt}])

In [ ]:
# Accessing the generated response from the AI model.
openai_response.choices[0].message.content

In [ ]:
# Interesting question:
# Did gpt-4o-mini **without context** give us a **correct** 2022 revenue value?

### Interpretation:
How is the LLM able to answer the same question now?

The LLM can now answer the question accurately because the relevant financial data is explicitly provided in the `retrieved_context`, allowing the model to reference it directly instead of relying on its pre-trained knowledge.

As you saw in the example above, we

- **retrieved** the context from an external source
- **augmented** our prompt that passes to the LLM, and
- **generated** the response

This is Retrieval Augmented Generation in a nutshell!

### Basic RAG app architecture

In the previous example, we ***manually*** retrieved the context from the given file which for all purposes is impractical!

Therefore, we have to devise a strategy that enables us to:

- Take the query from the user
- Identify the documents from the external source that might be relevant for the query.
- Pass those documents' information as context to the LLM
- LLM then generates the final response

To do the above, we can follow a simple standard architecture as shown below (Image source - https://huyenchip.com/2024/07/25/genai-platform.html)

<center><img src="https://huyenchip.com/assets/pics/genai-platform/3-rag.png" width=500 height=400/></center>

As you can see in the above image, the retriever would be the key component of this entire architecture.

To build the retriever, we have to follow these steps:

- Connect to the document source
- Break the documents down to manageable chunks.
   - This is due to the fact that taking in the entire document source for building the context will exceed the token limits of the LLM.
   - This process is also called **Chunking**.
- Perform a search for the most relevant chunks based on the given query.
- Pass those relevant chunks to the LLM.

For performing the search or retrieval process, we will be following an **embedding-based approach.**

<center><img src="https://cdn.prod.website-files.com/640248e1fd70b63c09bd3d09/653fd23f1565c0c1da063efc_Semantic%20Search%20Text%20Embeddings%20(1).png" width =500/></center>

### Understanding Embedding based approach

In the embedding based approach:

- We convert the document chunks in the database to vector embeddings and store it in a vector store.

- Convert the given user query to an embedding.

- Find the document chunks whose vector embeddings are closest to the given query embedding using a vector search algorithm like FAISS (Facebook AI Similarity Search)

<center><img src="https://miro.medium.com/v2/resize:fit:1400/1*h_btyitJX79d-gFE8RaMQg.png" width=500/></center>

### Tools for building the RAG App

Now that we are familiar with the overall architecture, we can now go ahead and structure the tools that we'll use for the upcoming demonstration:

- OpenAI LLM (model - GPT 4o-mini): This will be our primary model for generating the responses
- LangChain: Langchain is a powerful framework for orchestrating different layers in the RAG app. We shall use this to build the retriever end-to-end and also connect with other tools for tasks such as
    - Chunking - RecursiveCharacterTextSplitter
    - Embedding Model - OpenAIEmbeddings
    - Vector Search Model - FAISS
- Gradio: This will help in building a simple UI at the end.

## Problem Statement
Shopping online can be overwhelming. You search for a simple pair of shoes, but end up scrolling through countless options—many irrelevant, some too expensive, others just not right. Traditional search engines rely on keywords, often missing what you truly need.

Let's build an AI-powered product discovery chatbot changes this. Using advanced language models and vector-based search, it goes beyond keywords to understand your intent, offering personalized, context-aware recommendations in seconds.

<center><img src="https://www.pranathiss.com/static/assets/images/ai-powered-chatBot.webp" width=500/></center>

This smart solution enhances the shopping experience, increasing customer satisfaction, engagement, and conversions. The future of e-commerce is here—smarter, intuitive, and built for you.

### Dataset Used:
The given dataset contains information about various products, including their IDs, descriptions, and specifications. Below is a detailed description of each column and the type of data it contains.

You can download the entire dataset [here](https://www.kaggle.com/datasets/piyushjain16/amazon-product-data).
Or you can download the smaller sample dataset [here](https://drive.google.com/file/d/1ohd9xo19HmDVIwpXPf_IyMkwr29gJJxR/view?usp=drive_link).

#### Column Descriptions:
- `PRODUCT_ID (Integer)`
A unique identifier assigned to each product.
Example: 1925202, 2673191
- `TITLE (String)`
The name or title of the product, usually a brief summary.
Example: "ArtzFolio Tulip Flowers Blackout Curtain for D...",
"Marks & Spencer Girls' Pyjama Sets T86_2561C_N..."
- `BULLET_POINTS (List of Strings / NaN)`
A list of key product features and benefits in bullet format.
- `DESCRIPTION (String / NaN)`
A detailed textual description of the product, including specifications, features, and usage instructions.
Example: "Specifications: Color: Red, Material: Aluminium..."
- `PRODUCT_TYPE_ID (Integer / NaN)`
A numeric identifier indicating the type or category of the product.
Example: 1650, 2996, 7537
- `PRODUCT_LENGTH (Float)`
The length of the product, likely measured in millimeters or inches.
Example: 2125.98, 393.7, 748.031495

### Steps:

1. **Data Preparation**:
   - Load and process the dataset using pandas

2. **Vector Store Setup**:
   - Convert product descriptions into embeddings.
   - Store embeddings in a vector database.

3. **Building the Chatbot**:
   - Use LangChain to create an LLM pipeline.
   - Develop a simple chatbot to answer product-related queries.

4. **Creating a UI**:
   - Implement a Gradio-based UI for user interaction.

In [ ]:
# Importing the KaggleHub library to interact with datasets and models available on Kaggle.
import kagglehub

# Importing the CSV module for reading and writing CSV files.
import csv

# Importing pandas for data manipulation and analysis.
import pandas as pd

# Importing numpy for numerical operations and handling arrays efficiently.
import numpy as np

# Importing os to interact with the operating system, such as environment variables and file paths.
import os

# Importing getpass to securely handle user input (e.g., API keys or passwords).
import getpass

import math

### STEP 1: Data Preparation

In [ ]:
# 1. Open the "files" panel by clicking on the "folder" icon on the left.
# 2. Click on ".." to go to the root.
# 3. Check you can see "contnet" directory. Expand it (there'll be some default content there)
# 4. Drag and drop the sample_dataset.csv to go *under the content/ folder*.

df = pd.read_csv("/content/sample_dataset.csv",index_col=0)

In [ ]:
df.head(20)

In [ ]:
print(df.to_csv(columns=['DESCRIPTION'], sep='\t', index=False))

**Constructing the text data**

It's useful to use both `Title` and `Description`. To help downstream models understand which content is title and which content is description, we will add a prefix explaining which section is title and which is description. So each row should look like

```
Title
{Title}
Description
{Description}
```

In [ ]:
## Let's construct the text data
# Initializing empty lists to store product descriptions and their lengths
product_description = []
product_description_len = []

# Iterating through each row in the dataframe df2
for row in df.iterrows():
    product = ""  # Initialize an empty string to accumulate product details

    # Extracting the product title from the current row
    title = row[1]["TITLE"]

    # Checking if the title is valid (not NaN or missing)
    if type(title) != float or not math.isnan(title):
        product += "Title\n" + title + "\n"  # Append the title to the product description

    # Extracting the product description from the current row
    description = row[1]["DESCRIPTION"]

    # Checking if the description is valid (not NaN or missing)
    if type(description) != float or not math.isnan(description):
        product += "Description\n" + description + "\n"  # Append the description to the product details

    # Check if either title or description was added
    added_content = title or description
    if added_content:
        product = product.strip()  # Remove any leading/trailing whitespace
        product_description.append(product)  # Add the formatted product details to the list
        product_description_len.append(len(product))  # Store the length of the product description


In [ ]:
# Checking the length of the data
print(f"Number of elements {len(product_description)}")

In [ ]:
# Check a sample product description data
print(product_description[2])

In [ ]:
type(product_description[0])

In [ ]:
# Print the total number of product descriptions processed
print("Number of items", len(product_description_len))

# Print the minimum length of the product descriptions
print("Min Length of the description:",np.min(product_description_len))

# Print the average (mean) length of the product descriptions
print("Avg Length of the description:",np.mean(product_description_len))

# Print the maximum length of the product descriptions
print("Max Length of the description:",np.max(product_description_len))

### Interpretation:

What does the above result signify about the data?


*   A minimum length of 18 suggests that some product descriptions might be too brief
*  With an average length of 385.9 characters, most product descriptions contain a reasonable amount of information



### STEP 2: Vector Store Setup

Let's try to get a few of the basic questions answered about vector stores before we start using it.

### What is a vector store?
A vector store is a specialized database that stores data in the form of numerical vectors, allowing efficient searching and retrieval based on similarity rather than exact matches.

### Why do we need a vector store?
Traditional databases rely on exact keyword matches, which can miss relevant information. A vector store helps find similar content by understanding relationships and meaning in data.

### How does a vector store work?
It converts text, images, or other data into numerical vectors using AI models, then stores these vectors and retrieves similar ones using techniques like cosine similarity.

### How does a vector store improve search results?
It enables searches based on meaning rather than just keywords, providing more relevant results even if the exact terms don't match.

### What are some popular vector store tools?
- FAISS (Facebook AI Similarity Search)
- Pinecone
- Weaviate
- Chroma